In [1]:
import dask
import matplotlib.pyplot as plt
import xarray as xr

from saidownscale import catalog
from saidownscale.downscaling_utils import interpolate_coarse_to_fine_grid
from saidownscale.qa_flags import DIR_QA_FLAG_CONSTANT_INPUTS

In [2]:
VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds", "hurs"]

# Set up cluster

In [3]:
import coiled
from frisky import hijack

cluster = coiled.Cluster(
    name="srm-qaqc-flags-nrh",
    region="us-west-2",
    n_workers=12,
    worker_vm_types=["m8gn.xlarge"],
    scheduler_vm_types="c8g.xlarge",
    spot_policy="spot_with_fallback",
    use_best_zone=True,
    tags={"Project": "SRM"},
    worker_options={"nthreads": 8},
    environ={"ZARR_ASYNC__CONCURRENCY": "128"},
)

client = hijack(cluster.get_client())
client

[2026-09-06 13:12:05,764][INFO    ][coiled] Fetching latest package priorities...
[2026-09-06 13:12:05,765][INFO    ][coiled.package_sync] Resolving your local /Users/clairezarakas/Documents/science/srm-downscaling/uv.lock Python environment...
[2026-09-06 13:12:05,875][INFO    ][coiled.package_sync] Scanning 293 python packages...
[2026-09-06 13:12:06,120][INFO    ][coiled] Running pip check...
[2026-09-06 13:12:06,422][INFO    ][coiled] Validating environment...
[2026-09-06 13:12:07,147][INFO    ][coiled] Creating wheel for ~/Documents/science/srm-downscaling/src...
[2026-09-06 13:12:07,233][INFO    ][coiled] Creating wheel for srm...
[2026-09-06 13:12:09,031][INFO    ][coiled] Uploading coiled_local_src...
[2026-09-06 13:12:10,048][INFO    ][coiled] Uploading srm...
[2026-09-06 13:12:10,953][INFO    ][coiled] Creating software environment...
[2026-09-06 13:12:13,782][INFO    ][coiled] Creating Cluster (name: srm-qaqc-flags-nrh, https://cloud.coiled.io/clusters/2014136 ). This usuall

<frisky.Client: scheduler="wss://cluster-uxgfa.dask.host/DbLJSRinwc596SdF/frisky-comm?__frisky_dial_host=35.94.150.85" id="client-0">

# rsds [lat, dayofyear]

Markdown was generated by Claude, then reviewed and edited.
The ceiling is built as a function of `(dayofyear, lat)`: the maximum `rsds` across all longitudes in that latitude band (a "zonal max"), taken across ERA5 and two GCMs (`CESM2-WACCM`, `UKESM`) and three scenario groups (`historical`, `g6_1p5k`, `ssp245`). Because it is the *union* of several models' and scenarios' extremes, further loosened by maxing over longitude, it is deliberately a high bar rather than a tight statistical bound. It only depends on the input data, not on which output is being checked, so it is cached rather than recomputed on every run.

This is an intentionally simpler, standalone check -- it is not connected to the rolling-window plausible-bounds check in `saidownscale.qaqc.calculate_reasonable_bounds_doy`. The evaluation currently covers one debiased output (`CESM2-WACCM`, `ssp245`, ensemble member `003`) as a first pass.

This is of dimensions [lat,dayofyear] and we are calculating a really high max bound: the maximum downwelling solar radiation at the surface across all longitudes in that zonal band, in observations and across historical and future scenarios.

This only needs to be done once. (Only needs to be rerun if the input data changes)

In [4]:
# Generic cache helpers: write/read a DataArray to/from one zarr store, keyed by a string group name. Nothing here is specific to `rsds`.


def load_cached(key: str, fpath: str) -> xr.DataArray | None:
    """None on a cache miss; the caller decides what to do about it."""
    try:
        return xr.open_zarr(fpath, group=key)["data"].load()
    except FileNotFoundError:
        return None


def save_cached(key: str, da: xr.DataArray, fpath: str) -> None:
    da.rename("data").to_dataset().to_zarr(fpath, group=key, mode="w")

In [5]:
def calculate_fine_gcm_zonal_limit(coarse_gcm_grid_tseries, fine_obs_doy):
    """
    For one GCM's coarse-resolution time series: reduce to a day-of-year max, regrid that onto the fine ERA5 grid with `interpolate_coarse_to_fine_grid`, then take the max across longitude to get that source's zonal day-of-year bound.
    """
    coarse_gcm_doy_grid = coarse_gcm_grid_tseries.groupby("time.dayofyear").max(dim="time")
    fine_gcm_doy_grid = interpolate_coarse_to_fine_grid(
        da_coarse_to_regrid=coarse_gcm_doy_grid, da_fine_grid=fine_obs_doy
    )
    fine_gcm_doy_zonal = fine_gcm_doy_grid.max(dim="lon")

    return fine_gcm_doy_zonal

In [6]:
def calculate_overall_rsds_limit(
    fine_obs: xr.DataArray,
    plot_gcm_limits: bool = False,
    gcm_list: list[str] = ["CESM2-WACCM6", "UKESM1-1-LL"],
) -> xr.DataArray:
    # The same reduction applied directly to ERA5, which is already on the fine grid so no regridding step is needed.
    fine_obs_doy = fine_obs.groupby("time.dayofyear").max(dim="time")
    fine_obs_doy_zonal = fine_obs_doy.max(dim="lon")

    zonal_limit_across_all = fine_obs_doy_zonal
    # Loop over all GCMs and scenarios, regrid each to the fine grid, and take the maximum across all of them.
    # This is a way of estimating what is physically possible given variation in solar radiation by latitude and day-of-year.
    # The idea is that the highest rsds, in obs or under any ensemble member of any GCM under any scenario is a reasonable upper bound for what is physically possible.
    for gcm in gcm_list:
        print(gcm)
        for scenario in ["historical", "g6_1p5k", "ssp245"]:
            print(scenario)
            gcm_scenario = catalog.get(gcm).to_xarray()[scenario]["rsds"]

            # Combine every source into one ceiling: for each GCM x scenario combination, compute that source's zonal day-of-year max with
            # the helper function, and keep a running elementwise maximum across all of them, starting from the ERA5 value. `zonal_limit_across_all`
            # is the final ceiling, committed to the cache in the next cell.
            gcm_zonal_limit = (
                calculate_fine_gcm_zonal_limit(gcm_scenario, fine_obs_doy=fine_obs_doy)
                .max(dim="ensemble_member")
                .load()
            )

            zonal_limit_across_all = zonal_limit_across_all.where(
                (zonal_limit_across_all > gcm_zonal_limit) | gcm_zonal_limit.isnull(),
                gcm_zonal_limit,
            ).load()

            if plot_gcm_limits:
                plt.figure()
                gcm_zonal_limit.plot()
                plt.show()
    return zonal_limit_across_all

In [ ]:
# Load ERA5 as the fine reference grid: it's both the observational input to the ceiling and the common fine grid every GCM below gets regridded onto, so all sources land on the same grid before combining.

fine_obs = catalog.get("ERA5").to_xarray()["rsds"]

zonal_limit_across_all = calculate_overall_rsds_limit(
    fine_obs=fine_obs,
    plot_gcm_limits=False,
    gcm_list=["CESM2-WACCM6", "UKESM1-1-LL"],
)

save_cached(
    key="zonal_doy_max_rsds",
    da=zonal_limit_across_all,
    fpath=DIR_QA_FLAG_CONSTANT_INPUTS + "zonal_doy_max_rsds.zarr",
)

# Outlier thresholds from obs

In [ ]:
from saidownscale import catalog
from saidownscale.qa_flags import calculate_thresholds

In [ ]:
obs = catalog.get("ERA5").to_xarray()
obs = obs[VARIABLES]

In [ ]:
from saidownscale.downscaling_utils import rechunk

obs_fine = xr.Dataset(
    {var: rechunk(da=obs[var], pattern="full_time") for var in obs.data_vars},
    attrs=obs.attrs,
)

In [ ]:
def calculate_stats_for_outliers(obs, subset=None, timescale="annual"):
    if subset is not None:
        obs_subset = obs.sel(lat=subset[0], lon=subset[1]).load()
    else:
        obs_subset = obs

    if timescale == "annual":
        annual = obs_subset.groupby("time.year")
        obs_annual_max, obs_annual_min = dask.compute(annual.max(), annual.min())

        obs_max = obs_annual_max.max(dim=["year"])
        obs_min = obs_annual_min.min(dim=["year"])

        obs_max_std = obs_annual_max.std(dim=["year"])
        obs_min_std = obs_annual_min.std(dim=["year"])

    elif timescale == "monthly":
        monthly = obs_subset.groupby(["time.year", "time.month"])
        obs_monthly_max, obs_monthly_min = dask.compute(monthly.max(), monthly.min())

        obs_max = obs_monthly_max.max(dim="year")
        obs_min = obs_monthly_min.min(dim="year")
        obs_max_std = obs_monthly_max.std(dim="year")
        obs_min_std = obs_monthly_min.std(dim="year")

    elif timescale == "dayofyear":
        rolling_doy_max = obs_subset.rolling(time=30, center=True).max()
        rolling_doy_min = obs_subset.rolling(time=30, center=True).min()

        obs_max = rolling_doy_max.groupby("time.dayofyear").max()
        obs_min = rolling_doy_min.groupby("time.dayofyear").min()
        obs_max_std = rolling_doy_max.groupby("time.dayofyear").std()
        obs_min_std = rolling_doy_min.groupby("time.dayofyear").std()

    return [obs_max, obs_min, obs_max_std, obs_min_std]

##### Annual stats

In [ ]:
[obs_max, obs_min, obs_max_std, obs_min_std] = calculate_stats_for_outliers(
    obs=obs_fine, subset=None, timescale="annual"
)
[outlier_thresh_low, outlier_thresh_high] = calculate_thresholds(
    obs_max, obs_min, obs_max_std, obs_min_std
)

In [ ]:
combined = xr.merge(
    [
        obs_max.rename({v: f"{v}_max" for v in obs_max.data_vars}),
        obs_min.rename({v: f"{v}_min" for v in obs_min.data_vars}),
        obs_max_std.rename({v: f"{v}_max_std" for v in obs_max_std.data_vars}),
        obs_min_std.rename({v: f"{v}_min_std" for v in obs_min_std.data_vars}),
    ]
)

combined.to_zarr(DIR_QA_FLAG_CONSTANT_INPUTS + "annual_obs_thresholds_global.zarr", mode="w")

##### Day of year stats

In [ ]:
[obs_max, obs_min, obs_max_std, obs_min_std] = calculate_stats_for_outliers(
    obs=obs_fine, subset=None, timescale="dayofyear"
)

In [ ]:
# Sometimes this crashes after a few variables -- works if you rerun
STORE = DIR_QA_FLAG_CONSTANT_INPUTS + "doy_obs_thresholds_global.zarr"

stats = {"max": obs_max, "min": obs_min, "max_std": obs_max_std, "min_std": obs_min_std}

for i, var in enumerate(VARIABLES):
    print(var)
    per_var = xr.merge(
        [
            stats["max"][[var]].rename({var: f"{var}_max"}),
            stats["min"][[var]].rename({var: f"{var}_min"}),
            stats["max_std"][[var]].rename({var: f"{var}_max_std"}),
            stats["min_std"][[var]].rename({var: f"{var}_min_std"}),
        ]
    )
    mode = "w" if i == 0 else "a"
    per_var.to_zarr(STORE, mode=mode)

# Save to coarse GCM grids

In [7]:
from saidownscale.downscaling_utils import interpolate_fine_to_coarse_grid

In [8]:
zonal_limit_across_all = load_cached(
    key="zonal_doy_max_rsds", fpath=DIR_QA_FLAG_CONSTANT_INPUTS + "zonal_doy_max_rsds.zarr"
)
for gcm in ["CESM2-WACCM6", "UKESM1-1-LL"]:
    grid_example = (
        catalog.get(gcm).to_xarray()["historical"]["rsds"].isel(ensemble_member=0, time=0)
    )
    zonal_limit_across_all_gcmgrid = interpolate_fine_to_coarse_grid(
        da_fine_to_coarsen=zonal_limit_across_all, da_coarse_grid=grid_example
    )
    save_cached(
        key="zonal_doy_max_rsds",
        da=zonal_limit_across_all_gcmgrid,
        fpath=DIR_QA_FLAG_CONSTANT_INPUTS + "zonal_doy_max_rsds_" + gcm + ".zarr",
    )

/Users/clairezarakas/Documents/science/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/clairezarakas/Documents/science/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
ds = xr.open_zarr(DIR_QA_FLAG_CONSTANT_INPUTS + "annual_obs_thresholds_global.zarr")

for gcm in ["CESM2-WACCM6", "UKESM1-1-LL"]:
    grid_example = (
        catalog.get(gcm).to_xarray()["historical"]["rsds"].isel(ensemble_member=0, time=0)
    )
    ds_coarse = xr.Dataset(
        {
            var: interpolate_fine_to_coarse_grid(
                da_fine_to_coarsen=ds[var], da_coarse_grid=grid_example
            )
            for var in ds.data_vars
        }
    )
    ds_coarse.to_zarr(DIR_QA_FLAG_CONSTANT_INPUTS + "annual_obs_thresholds_global_" + gcm + ".zarr")

/Users/clairezarakas/Documents/science/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/clairezarakas/Documents/science/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
ds = xr.open_zarr(DIR_QA_FLAG_CONSTANT_INPUTS + "doy_obs_thresholds_global.zarr")

for gcm in ["CESM2-WACCM6", "UKESM1-1-LL"]:
    grid_example = (
        catalog.get(gcm).to_xarray()["historical"]["rsds"].isel(ensemble_member=0, time=0)
    )
    ds_coarse = xr.Dataset(
        {
            var: interpolate_fine_to_coarse_grid(
                da_fine_to_coarsen=ds[var], da_coarse_grid=grid_example
            )
            for var in ds.data_vars
        }
    )
    ds_coarse.to_zarr(DIR_QA_FLAG_CONSTANT_INPUTS + "doy_obs_thresholds_global_" + gcm + ".zarr")

/Users/clairezarakas/Documents/science/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/clairezarakas/Documents/science/srm-downscaling/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [9]:
cluster.shutdown()

[2026-09-06 13:14:31,451][INFO    ][coiled] Cluster 2014136 deleted successfully.
